In [7]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

# 1. NẠP VÀ XỬ LÝ DỮ LIỆU
sales_training_data = pd.read_csv('sales.csv', parse_dates=['Date'])
sales_training_data = sales_training_data.sort_values('Date').reset_index(drop=True)

q99 = sales_training_data['Revenue'].quantile(0.99)
sales_training_data['Revenue_Smoothed'] = np.where(
    sales_training_data['Revenue'] > q99,
    q99,
    sales_training_data['Revenue']
)

# 2. TẠO ĐẶC TRƯNG
def extract_features(df):
    df = df.copy()
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['day_of_year'] = df['Date'].dt.dayofyear
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['Date'].dt.is_month_end.astype(int)
    df['is_double_day'] = (df['month'] == df['day']).astype(int)
    df['is_black_friday'] = ((df['month'] == 11) & (df['day'] >= 23) & (df['day'] <= 28)).astype(int)
    df['is_tet'] = df['month'].isin([1, 2]).astype(int)
    df['time_index'] = (df['Date'] - df['Date'].min()).dt.days
    return df

training_features = extract_features(sales_training_data)
features = ['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year',
            'month_sin', 'month_cos', 'is_weekend', 'is_month_start', 'is_month_end',
            'is_double_day', 'is_black_friday', 'is_tet', 'time_index']

X_train = training_features[features]
y_train_log = np.log1p(training_features['Revenue_Smoothed'])

# 3. CHẠY KIỂM ĐỊNH CHÉO (CROSS-VALIDATION)
print("--- KẾT QUẢ CROSS-VALIDATION (TIME SERIES SPLIT) ---")
timeSeriesSplitter = TimeSeriesSplit(n_splits=5)
fold_number = 1
cv_mae_scores = []

for train_index, validation_index in timeSeriesSplitter.split(X_train):
    X_train_fold = X_train.iloc[train_index]
    X_validation_fold = X_train.iloc[validation_index]
    y_train_fold = y_train_log.iloc[train_index]
    y_validation_fold = y_train_log.iloc[validation_index]

    cv_model = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.01, num_leaves=31,
        objective='huber', random_state=42, n_jobs=-1, verbose=-1
    )
    cv_model.fit(X_train_fold, y_train_fold)

    validation_predictions = np.expm1(cv_model.predict(X_validation_fold))
    y_validation_real = np.expm1(y_validation_fold)

    fold_mae = mean_absolute_error(y_validation_real, validation_predictions)
    cv_mae_scores.append(fold_mae)
    print(f"Fold {fold_number} - MAE: {fold_mae:,.2f}")
    fold_number += 1

print(f"-> Trung bình MAE: {np.mean(cv_mae_scores):,.2f}")

--- KẾT QUẢ CROSS-VALIDATION (TIME SERIES SPLIT) ---
Fold 1 - MAE: 1,163,586.59
Fold 2 - MAE: 1,182,868.30
Fold 3 - MAE: 1,262,152.72
Fold 4 - MAE: 825,619.94
Fold 5 - MAE: 636,187.96
-> Trung bình MAE: 1,014,083.10
